In [0]:
dbutils.widgets.removeAll()


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsssmartdata2698")


In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/olist_customers_dataset.csv"

In [0]:
df_customers = spark.read.option("header", True)\
                         .option("inferSchema", True)\
                         .csv(ruta)

In [0]:
customers_schema = StructType(fields=[
    StructField("customer_id", StringType(), True),
    StructField("customer_unique_id", StringType(), True),
    StructField("customer_zip_code_prefix", IntegerType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True)
])

In [0]:
df_customers_final = spark.read\
.option("header", True)\
.schema(customers_schema)\
.csv(ruta)

In [0]:
customers_selected_df = df_customers_final.select(
    col("customer_id"),
    col("customer_unique_id"),
    col("customer_zip_code_prefix"),
    col("customer_city"),
    col("customer_state")
)

In [0]:
customers_renamed_df = customers_selected_df


In [0]:
customers_final_df = customers_renamed_df.withColumn("ingestion_date", current_timestamp())



In [0]:
customers_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.customers_raw")